# Lekcija 1: Slike kao polja

Digitalna slika je dvodimenzionalna mreža brojeva. U ovoj ćemo lekciji pomoću biblioteke NumPy izraditi malu sliku od početka, proučiti kako se prikazuju slike u sivim tonovima i slike u boji te ih prikazati pomoću biblioteke Matplotlib.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Slika u sivim tonovima je dvodimenzionalno polje

Svaki element predstavlja intenzitet piksela. Za 8-bitnu sliku vrijednosti piksela kreću se od 0 (crno) do 255 (bijelo).


In [ ]:
im = np.zeros((100, 100), dtype=np.uint8)
im[20:80, 20:80] = 255   # a white square on a black background

print('shape:', im.shape, 'dtype:', im.dtype)

plt.imshow(im, cmap='gray', vmin=0, vmax=255)
plt.title('Grayscale image (100x100)')
plt.axis('off')
plt.show()

## Jednostavne operacije na razini piksela

Budući da je slika samo polje brojeva, standardne operacije biblioteke NumPy mogu se primijeniti izravno, primjerice za invertiranje intenziteta.


In [ ]:
im_inv = 255 - im

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(im, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(im_inv, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Inverted (255 - pixel)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Slika u boji je trodimenzionalno polje

Dimenzije slike su `(visina, širina, kanali)` &mdash; to je rezultat izraza `array.shape`. **Prvom** osi odabire se redak (kretanje prema dolje po slici, odnosno $y$), **drugom** osi odabire se stupac (kretanje vodoravno po slici, odnosno $x$), a **trećom** osi odabire se kanal boje (crveni, zeleni ili plavi). Prema uobičajenoj konvenciji, ishodište se nalazi u gornjem lijevom kutu.

**Upozorenje:** Iako `array.shape` i indeksiranje u biblioteci NumPy koriste redoslijed `(redak, stupac)` = `(y, x)`, mnoge funkcije biblioteke OpenCV koje primaju točke ili dimenzije &mdash; `cv2.circle`, `cv2.rectangle`, `cv2.resize`, argument `dsize` funkcije `cv2.warpAffine` itd. &mdash; očekuju obrnut redoslijed, odnosno `(x, y)` ili `(širina, visina)`. Ta zamjena redoslijeda čest je izvor pogrešaka; uvijek provjerite redoslijed argumenata funkcije umjesto da pretpostavite da odgovara redoslijedu dimenzija polja.


In [ ]:
imc = np.zeros((50, 100, 3), dtype=np.uint8)
imc[:, :, 0] = 255        # red channel on for the whole image
imc[10:40, 20:80, 1] = 255  # add green in the middle -> red + green looks yellow

print('shape:', imc.shape, 'dtype:', imc.dtype)

plt.imshow(imc)
plt.title('Color image (100x100x3)')
plt.axis('off')
plt.show()

## Kanali boje obično su u redoslijedu RGB

Tri kanala boje obično su poredana ovim redoslijedom: crveni, zeleni i plavi (RGB). Kako bismo to ilustrirali, izradit ćemo malu sliku od **šest psihološki osnovnih boja**, raspoređenih u mrežu s 2 retka i 3 stupca. To su šest boja (crna, bijela, crvena, žuta, zelena i plava) koje ljudski vidni sustav doživljava kao osnovne, nepomiješane boje.


In [ ]:
im_primaries = np.zeros((2, 3, 3), dtype=np.uint8)
im_primaries[0, 0, :] = (0, 0, 0)  # black
im_primaries[0, 1, :] = (255, 255, 255)  # white
im_primaries[0, 2, :] = (255, 0, 0)  # red
im_primaries[1, 0, :] = (255, 255, 0)  # yellow
im_primaries[1, 1, :] = (0, 255, 0)  # green
im_primaries[1, 2, :] = (0, 0, 255)  # blue
print('shape:', im_primaries.shape, '  (2 rows, 3 columns, 3 color channels)')

plt.imshow(im_primaries)
plt.title('The six psychological primaries')
plt.axis('off')
plt.show()

Izdvajanje raspona indeksa (engl. *slicing*) po **prvoj** osi odabire cijele retke (vodoravni pojas slike), a po **drugoj** osi cijele stupce (okomiti pojas).


In [ ]:
top_row = im_primaries[0:1, :, :]        # first axis sliced -> a full-width horizontal band
left_column = im_primaries[:, 0:1, :]    # second axis sliced -> a full-height vertical band
# NOTE: a bare integer index (im_primaries[0, :, :]) would DROP that axis entirely, turning
# shape (1, 3, 3) into (3, 3) -- imshow would then misread it as a 3x3 grayscale image instead
# of a 1-row RGB strip. Slicing with 0:1 keeps the axis alive as size 1.

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(im_primaries)
axes[0].set_title('Full image', fontsize=9)
axes[1].imshow(top_row)
axes[1].set_title(f'Top row  [0:1, :, :]', fontsize=9)
axes[2].imshow(left_column)
axes[2].set_title(f'Left column  [:, 0:1, :]', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Učitavanje stvarne fotografije: BGR i RGB

Sve dosadašnje slike u boji ručno su izrađene s kanalima u redoslijedu RGB &mdash; upravo onako kako očekuje funkcija `plt.imshow` iz biblioteke Matplotlib. Stvarne fotografije, međutim, učitavamo funkcijom `cv2.imread` iz biblioteke OpenCV, koja čita (i zapisuje) slike u boji s kanalima u redoslijedu **BGR** &mdash; plavi je prvi, a crveni posljednji &mdash; obrnuto od redoslijeda RGB koji pretpostavljaju sve ostale Python biblioteke za obradu slika i crtanje. Ako BGR polje izravno proslijedimo funkciji `plt.imshow`, ona će pogrešno protumačiti kanale boje, što će rezultirati neobičnim prikazom.


In [ ]:
im_bgr = cv2.imread('../img/rose.jpg')
print('shape:', im_bgr.shape, ' (last axis is in BGR order)')

plt.imshow(im_bgr)
plt.title('im_bgr shown directly -- wrong colors!')
plt.axis('off')
plt.show()

### Dva načina za rješavanje ovog problema

1. **Obrnuti redoslijed kanala pomoću indeksiranja u biblioteci NumPy.** Izraz `img[:, :, ::-1]` obrće redoslijed posljednje osi, pretvarajući `[B, G, R]` u `[R, G, B]` &mdash; riječ je o običnoj operaciji nad poljem, bez potrebe za funkcijom biblioteke OpenCV.
2. **`cv2.cvtColor`.** Funkcija biblioteke OpenCV opće namjene za pretvorbu boja; `cv2.COLOR_BGR2RGB` izvodi potpuno istu zamjenu kanala, ali jasnije izražava *zašto* se ona izvodi te predstavlja uobičajen način pisanja takve pretvorbe u OpenCV kodu. Ista funkcija podržava i pretvorbe koje nisu jednostavno obrtanje redoslijeda, primjerice u sive tonove ili HSV, što ćemo obraditi u kasnijim lekcijama.

Oba bi postupka ovdje trebala dati identične rezultate jer je pretvorba BGR&rarr;RGB samo obrtanje redoslijeda triju kanala.


In [ ]:
im_rgb1 = im_bgr[:, :, ::-1]
im_rgb2 = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)

print('the two methods agree exactly:', np.array_equal(im_rgb1, im_rgb2))

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, im, title in zip(axes, [im_bgr, im_rgb1, im_rgb2],
                          ['im_bgr (wrong)', 'img[:, :, ::-1]', "cv2.cvtColor(..., BGR2RGB)"]):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: <a href="https://picryl.com" target="_blank" rel="noopener">Picryl</a></p>


### Zadaci

1. Izmijenite sliku u boji `imc` tako da pravokutnik bude cijan umjesto žut. Koji kanal ili kanale trebate promijeniti?
2. U polju `im_primaries` izdvojite *donji desni* blok (plavi) pomoću raspona indeksa redaka i stupaca te funkcijom `np.array_equal` provjerite podudara li se s novim blokom ispunjenim vrijednostima `(0, 0, 255)`.
3. Učitajte `rose.jpg` te ispišite `im_bgr[0, 0, :]` (gornji lijevi piksel, redoslijed BGR) zajedno s `im_rgb2[0, 0, :]` (redoslijed RGB). Ručno provjerite radi li se o istim trima brojevima, samo drukčije poredanima.
